In [1]:
import os, re, json, pickle
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

FAKE_PATH = "News_dataset/Fake.csv"
TRUE_PATH = "News_dataset/True.csv"

OUTPUT_CLEAN_CSV = "cleaned_dataset.csv"
OUTPUT_TOKENIZER = "tokenizer.pkl"
OUTPUT_EMB_MATRIX = "embedding_matrix.npy"
EMBED_DIM = 300
VOCAB_SIZE = 50000      
MAX_LEN_CAP = 500         

STOPWORDS = {
    'a','an','the','and','or','but','if','while','of','at','by','for','with','about','against',
    'between','into','through','during','before','after','to','from','up','down','in','out',
    'on','off','over','under','again','further','then','once','here','there','when','where',
    'why','how','all','any','both','each','few','more','most','other','some','such','no',
    'nor','not','only','own','same','so','than','too','very','can','will','just','don','should',
    'now','is','am','are','was','were','be','been','being','have','has','had','do','does','did'
}
print("The first step is done")

The first step is done


In [2]:
df_fake = pd.read_csv(FAKE_PATH)
df_true = pd.read_csv(TRUE_PATH)

for col in ['title', 'text']:
    if col not in df_fake.columns:
        raise ValueError(f"Fake.csv missing column: {col}")
    if col not in df_true.columns:
        raise ValueError(f"True.csv missing column: {col}")
    
def keep_or_set_subject(df):
    if "subject" in df.columns:
        return df[["title", "text", "subject"]].copy()
    out = df[["title", "text"]].copy()
    out["subject"] = "unknown"
    return out    

df_fake = keep_or_set_subject(df_fake)
df_true = keep_or_set_subject(df_true)

df_fake['label'] = 'FAKE'
df_true['label'] = 'REAL'

df = pd.concat([df_fake, df_true], ignore_index=True)
print("Merged shape:", df.shape)
df.head()


C:\Users\G14\AppData\Local\Temp\ipykernel_24624\781469885.py:1: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fake = pd.read_csv(FAKE_PATH)


Merged shape: (44919, 4)


,title,text,subject,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,FAKE
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,FAKE
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,FAKE
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,FAKE
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,FAKE


In [3]:
TAG_RE = re.compile(r"<[^>]+>")

def basic_clean(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = TAG_RE.sub(" ", s)                
    s = re.sub(r"[^a-z\s]", " ", s)       
    s = re.sub(r"\s+", " ", s).strip()  
    return s

def remove_stopwords(s: str) -> str:
    return " ".join([w for w in s.split() if w not in STOPWORDS])

df["title"] = df["title"].fillna("").astype(str)
df["text"]  = df["text"].fillna("").astype(str)

df["clean_title"] = df["title"].apply(basic_clean).apply(remove_stopwords)
df["clean_text"]  = df["text"].apply(basic_clean).apply(remove_stopwords)

df["clean_join"] = (df["clean_title"] + " " + df["clean_text"]).str.strip()

before = len(df)
df = df[df["clean_join"].str.len() > 0].reset_index(drop=True)
print(f"After cleaning: {df.shape} (dropped {before - len(df)} empty rows)")
df[["title", "subject", "label"]].head()


After cleaning: (44919, 7) (dropped 0 empty rows)


,title,subject,label
0,Donald Trump Sends Out Embarrassing New Year’...,News,FAKE
1,Drunk Bragging Trump Staffer Started Russian ...,News,FAKE
2,Sheriff David Clarke Becomes An Internet Joke...,News,FAKE
3,Trump Is So Obsessed He Even Has Obama’s Name...,News,FAKE
4,Pope Francis Just Called Out Donald Trump Dur...,News,FAKE


In [4]:
texts = df["clean_join"].tolist()

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

seqs = tokenizer.texts_to_sequences(texts)
lengths = np.array([len(s) for s in seqs if len(s) > 0])

if len(lengths) == 0:
    raise ValueError("Tokenization give the null string => check error")

p95 = int(np.percentile(lengths, 95))
max_len = int(min(p95, MAX_LEN_CAP))
print(f"95th percentile length = {p95} -> max_len = {max_len}")

padded = pad_sequences(seqs, maxlen=max_len, padding="post", truncating="post")
print("padded shape:", padded.shape)

with open(OUTPUT_TOKENIZER, "wb") as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer has been saved: {OUTPUT_TOKENIZER} (actual vocab = {len(tokenizer.word_index)+1})")


95th percentile length = 631 -> max_len = 500
padded shape: (44919, 500)
Tokenizer has been saved: tokenizer.pkl (actual vocab = 115466)


In [7]:
GLOVE_PATH = "wiki_giga_2024_300_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt"

def build_glove_matrix(glove_path, tokenizer, vocab_size=VOCAB_SIZE, emb_dim=EMBED_DIM):
    print("Loading GloVe vectors (It might take some minutes)...")
    embeddings_index = {}
    with open(glove_path, "r", encoding="utf-8") as f:
        for line in f:
            values = line.rstrip().split(" ")
            word = values[0]
            coefs = np.asarray(values[1:], dtype="float32")
            embeddings_index[word] = coefs

    word_index = tokenizer.word_index
    num_words = min(vocab_size, len(word_index) + 1)
    emb_matrix = np.zeros((num_words, emb_dim), dtype="float32")

    hits = 0
    for word, i in word_index.items():
        if i >= num_words:
            continue
        vec = embeddings_index.get(word)
        if vec is not None and vec.shape[0] == emb_dim:
            emb_matrix[i] = vec
            hits += 1
    print(f"GloVe coverage: {hits}/{num_words} = {hits/num_words:.2%}")
    return emb_matrix

if os.path.exists(GLOVE_PATH):
    emb_matrix = build_glove_matrix(GLOVE_PATH, tokenizer, VOCAB_SIZE, EMBED_DIM)
    np.save(OUTPUT_EMB_MATRIX, emb_matrix)
    print(f"Embedding matrix has been saved: {OUTPUT_EMB_MATRIX}, shape={emb_matrix.shape}")
else:
    print("Error!")


Loading GloVe vectors (It might take some minutes)...
GloVe coverage: 47269/50000 = 94.54%
Embedding matrix has been saved: embedding_matrix.npy, shape=(50000, 300)


In [8]:
out_cols = ["title", "text", "subject", "label", "clean_join"]
df[out_cols].to_csv(OUTPUT_CLEAN_CSV, index=False)
print(f"Cleaned data has been saved: {OUTPUT_CLEAN_CSV}")
df[out_cols].head()

Cleaned data has been saved: cleaned_dataset.csv


,title,text,subject,label,clean_join
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,FAKE,donald trump sends embarrassing new year s eve...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,FAKE,drunk bragging trump staffer started russian c...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,FAKE,sheriff david clarke becomes internet joke thr...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,FAKE,trump obsessed he even obama s name coded his ...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,FAKE,pope francis called donald trump his christmas...


In [10]:

meta = {
    "max_len": int(max_len),
    "vocab_size_limit": int(VOCAB_SIZE),
    "actual_vocab_size": int(len(tokenizer.word_index) + 1),
    "embedding_dim": int(EMBED_DIM),
    "using_glove": os.path.exists("glove.6B.300d.txt"),
    "rows": int(len(df)),
    "has_subject": True
}
with open("preprocess_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("ℹ️ preprocess_meta.json:", meta)
print("\n📊 Label distribution:")
print(df["label"].value_counts())
print("\n📊 Subject (top 10):")
print(df["subject"].value_counts().head(10))
print("\n✅ Pipeline is finished.")


ℹ️ preprocess_meta.json: {'max_len': 500, 'vocab_size_limit': 50000, 'actual_vocab_size': 115466, 'embedding_dim': 300, 'using_glove': False, 'rows': 44919, 'has_subject': True}

📊 Label distribution:
label
FAKE    23502
REAL    21417
Name: count, dtype: int64

📊 Subject (top 10):
subject
politicsNews                                                                             11272
worldnews                                                                                10145
News                                                                                      9050
politics                                                                                  6838
left-news                                                                                 4457
Government News                                                                           1570
US_News                                                                                    775
Middle-east                                  